In [1]:
from openai import OpenAI
from google import genai
import json
import os

In [2]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])









KeyError: 'GEMINI_API_KEY'

In [8]:
from google.genai import types

def analyze_audio_with_gemini(audio_path):

    # Read audio file
    with open(audio_path, "rb") as f:
        audio_bytes = f.read()

    prompt = """
    You are an AI compliance and forensic conversation analysis system for a banking institution.
        
    Analyze the provided audio conversation between a bank analyst and a customer.
        
    Your tasks are:
        
    1. Detect the total number of speakers.
    2. Identify each speaker's role and probable gender (only if clearly inferable).
    3. Provide timestamps for each spoken segment.
    4. Provide a concise but detailed summary.
    5. Detect compliance violations and anomalies.
    6. For every anomaly, include the exact timestamp range where it occurred.

    Compliance Rules to Check:
    - Request for sensitive personal information:
        • ATM PIN
        • CVV
        • OTP
        • Full debit/credit card number
        • Internet banking password
        • Account number (if requested without proper authentication protocol)
    - Attempt to improperly extract confidential information
    - Asking for personal contact details outside official channel
    - Inappropriate / unprofessional behavior
    - Aggression / threatening tone / harassment
    - Social engineering indicators
    
    Important:
    - Focus especially on the behavior of the bank analyst.
    - Be objective and evidence-based.
    - Do NOT hallucinate violations.
    - If nothing is detected, explicitly return empty anomalies list.
    
    Return STRICT JSON in this format:
    
    {
      "number_of_speakers": <integer>,
      "speaker_details": [
          {
              "speaker": "Speaker 1",
              "role": "<analyst / customer / unknown>",
              "gender": "<male / female / unknown>"
          },
          {
              "speaker": "Speaker 2",
              "role": "<analyst / customer / unknown>",
              "gender": "<male / female / unknown>"
          }
      ],
      "conversation_timeline": [
          {
              "timestamp_start": "HH:MM:SS",
              "timestamp_end": "HH:MM:SS",
              "speaker": "Speaker 1 or Speaker 2",
              "text": "<exact spoken content>"
          }
      ],
      "conversation_summary": "<concise summary>",
      "anomalies_detected": [
          {
              "type": "<sensitive_info_request / inappropriate_behavior / aggression / social_engineering / compliance_violation>",
              "description": "<clear explanation>",
              "speaker": "<Speaker 1 or Speaker 2>",
              "timestamp_start": "HH:MM:SS",
              "timestamp_end": "HH:MM:SS",
              "severity": "<low / medium / high>"
          }
      ]
    }
    
    If no anomalies are detected:
    {
      "number_of_speakers": <integer>,
      "speaker_details": [...],
      "conversation_timeline": [...],
      "conversation_summary": "<summary>",
      "anomalies_detected": []
    }
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[
            types.Part.from_bytes(
                data=audio_bytes,
                mime_type="audio/mp3"
            ),
            prompt
        ]
      
    )

    return response.text

output = analyze_audio_with_gemini("realistic_bank_call.mp3")
print(output)

```json
{
  "number_of_speakers": 2,
  "speaker_details": [
    {
      "speaker": "Speaker 1",
      "role": "analyst",
      "gender": "male"
    },
    {
      "speaker": "Speaker 2",
      "role": "customer",
      "gender": "female"
    }
  ],
  "conversation_timeline": [
    {
      "timestamp_start": "00:00:00",
      "timestamp_end": "00:00:05",
      "speaker": "Speaker 1",
      "text": "Good afternoon. I am Rahul from Bank Security team. Apka account verify karna hai."
    },
    {
      "timestamp_start": "00:00:06",
      "timestamp_end": "00:00:09",
      "speaker": "Speaker 2",
      "text": "Ji, kaunsi verification? Mujhe koi alert nahi mila."
    },
    {
      "timestamp_start": "00:00:09",
      "timestamp_end": "00:00:16",
      "speaker": "Speaker 1",
      "text": "System ne suspicious activity show ki hai. We need to confirm. Aap apna account number bata sakte hain?"
    },
    {
      "timestamp_start": "00:00:16",
      "timestamp_end": "00:00:19",
      "speak

In [2]:
from elevenlabs import ElevenLabs

# 🔐 Paste your API key inside the quotes
client = ElevenLabs(api_key="sk_1635e109bce60e8f9eaa7f5da342ab9b957051fa9d40918e")

print("Connected to ElevenLabs successfully!")

Connected to ElevenLabs successfully!


In [13]:
from elevenlabs.client import ElevenLabs
from elevenlabs import save

client = ElevenLabs(api_key="sk_1635e109bce60e8f9eaa7f5da342ab9b957051fa9d40918e")

analyst_voice = "IKne3meq5aSn9XLyUdCD"   # Charlie (premade)
customer_voice = "EXAVITQu4vr4xnSDxMaL"  # Sarah (premade)

conversation = [
    ("analyst", "Good afternoon, I am rahul from bank security team. Aapka account verify karna hai."),

    ("customer", "Ji, kaun si verification? Mujhe koi alert nhi mila."),

    ("analyst", "System ne suspicious activity show ki hai. We need to confirm. Aap apna account number bata sakti hain?"),

    ("customer", "Account number kyun chahiye? This is sensitive information."),

    ("analyst", "Haan ji, bas security check ke liye. Jaldi process complete ho jayega."),

    ("customer", "Okay, but I won't share my account details."),

    ("analyst", "Koi baat nahi. Ek OTP aayega, please woh confirm kar dena."),

    ("customer", "why do you need my otp ?"),

    ("analyst", "Okay. Waise, kya main aapse personal number le sakta hoon? Direct contact easier rahega."),

    ("customer", "Mera personal number? Bank call mein yeh inappropriate hai."),

    ("analyst", "Bas convenience ke liye puch raha tha."),

    ("customer", "Nahi. That's very unprofeessional. Main call end kar rahi hoon.")
]

with open("realistic_bank_call.mp3", "wb") as final_file:
    for speaker, text in conversation:
        voice_id = analyst_voice if speaker == "analyst" else customer_voice
        
        audio = client.text_to_speech.convert(
            voice_id=voice_id,
            model_id="eleven_turbo_v2",
            text=text
        )
        
        save(audio, "temp.mp3")
        
        with open("temp.mp3", "rb") as temp:
            final_file.write(temp.read())

print("Realistic 2-voice call generated successfully!")

Realistic 2-voice call generated successfully!


In [14]:
from IPython.display import Audio
Audio("realistic_bank_call.mp3")